# NB_patient_360

Builds `<catalog>.gold.patient_360` per the Silver to Gold STTM: one row per patient. The driving table is `silver.patients`, LEFT JOINed to hospitals, insurance providers and lab results aggregated per patient. Load type: FULL (CREATE OR REPLACE TABLE ... AS SELECT).

In [0]:
dbutils.widgets.text("catalog", "caresync_databricks")
CATALOG = dbutils.widgets.get("catalog")

SILVER = f"{CATALOG}.silver"
TARGET_TABLE = f"{CATALOG}.gold.patient_360"
print(f"Source: {SILVER}  ->  Target: {TARGET_TABLE}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {TARGET_TABLE}
COMMENT 'Gold: one row per patient with hospital, insurance and lab-test summary. Source: STTM Silver to Gold v1.0.'
AS
WITH lab_agg AS (
    SELECT
        patient_id,
        COUNT(*)                              AS total_tests,
        COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
        COUNT_IF(result_status = 'Critical')  AS critical_test_count,
        MAX(test_date)                        AS last_test_date
    FROM {SILVER}.lab_results
    GROUP BY patient_id
)
SELECT
    CAST(p.patient_id AS INT)                                                   AS patient_id,
    p.first_name,
    p.last_name,
    CAST(p.date_of_birth AS DATE)                                               AS date_of_birth,
    CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
    p.gender,
    p.city,
    p.state,
    CAST(p.registered_hospital_id AS INT)                                       AS hospital_id,
    h.hospital_name,
    CAST(p.insurance_provider_id AS INT)                                        AS insurance_provider_id,
    ip.provider_name                                                            AS insurance_provider_name,
    CAST(COALESCE(l.total_tests, 0) AS INT)                                     AS total_tests,
    CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                             AS abnormal_test_count,
    CAST(COALESCE(l.critical_test_count, 0) AS INT)                             AS critical_test_count,
    l.last_test_date                                                            AS last_test_date
FROM {SILVER}.patients p
LEFT JOIN {SILVER}.hospitals h
       ON p.registered_hospital_id = h.hospital_id
LEFT JOIN {SILVER}.insurance_providers ip
       ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg l
       ON p.patient_id = l.patient_id
""")

print(f"Written {spark.table(TARGET_TABLE).count():,} rows to {TARGET_TABLE}")

In [0]:
import json

record_count = spark.table(TARGET_TABLE).count()

# Returned as the notebook output (visible to Jobs / ADF / dbutils.notebook.run callers)
dbutils.notebook.exit(str(record_count))